# 06 — Results Analysis & Visualization

Cross-dataset comparison of all models:
- **Classical ML** (SVM, LogReg, Naive Bayes)
- **Fine-tuned BERT** (gbert-base / bert-base-uncased)
- **LLM** (GPT-4o, GPT-4o-mini)

Evaluated on German Sentiment (1,490 test) and Yelp Reviews (5,000 test).

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid", font_scale=1.1)

METRICS_DIR = Path("../results/metrics")
LABEL_NAMES = ["negative", "neutral", "positive"]

## 1. Load All Metrics

In [ ]:
# German Sentiment results
german_files = {
    "BERT (gbert-base)": "bert_finetuned_metrics.json",
    "SVM (TF-IDF)": "classical_svm.json",
    "LogReg (TF-IDF)": "classical_logistic_regression.json",
    "Naive Bayes (TF-IDF)": "classical_naive_bayes.json",
    "GPT-4o-mini (zero-shot)": "llm_gpt-4o-mini_zero_shot.json",
    "GPT-4o (zero-shot)": "llm_gpt-4o_zero_shot.json",
    "GPT-4o (few-shot)": "llm_gpt-4o_few_shot.json",
    "GPT-4o-mini (few-shot)": "llm_gpt-4o-mini_few_shot.json",
}

# Yelp Reviews results
yelp_files = {
    "BERT (bert-base-uncased)": "bert_yelp_finetuned_metrics.json",
    "SVM (TF-IDF)": "classical_svm_yelp.json",
    "LogReg (TF-IDF)": "classical_logistic_regression_yelp.json",
    "Naive Bayes (TF-IDF)": "classical_naive_bayes_yelp.json",
}

def load_results(file_map):
    rows = []
    raw = {}
    for name, fname in file_map.items():
        fpath = METRICS_DIR / fname
        if not fpath.exists():
            print(f"  SKIP {fname} (not found)")
            continue
        with open(fpath) as f:
            data = json.load(f)
        raw[name] = data
        test = data.get("test", {})
        latency = data.get("latency", {})
        rows.append({
            "Model": name,
            "F1 (weighted)": test.get("f1_weighted", 0),
            "Accuracy": test.get("accuracy", 0),
            "F1 (macro)": test.get("f1_macro", 0),
            "Precision": test.get("precision_weighted", 0),
            "Recall": test.get("recall_weighted", 0),
            "ROC-AUC": test.get("roc_auc_weighted") or 0,
            "Latency (ms)": latency.get("per_sample_ms", 0),
        })
    return pd.DataFrame(rows).sort_values("F1 (weighted)", ascending=False), raw

df_german, raw_german = load_results(german_files)
df_yelp, raw_yelp = load_results(yelp_files)

print(f"German: {len(df_german)} models loaded")
print(f"Yelp:   {len(df_yelp)} models loaded")
df_german

## 2. F1 Score Comparison — Both Datasets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# German
colors_german = ["#10B981" if "BERT" in m else "#3B82F6" if "TF-IDF" in m else "#8B5CF6" for m in df_german["Model"]]
axes[0].barh(df_german["Model"], df_german["F1 (weighted)"], color=colors_german)
axes[0].set_xlim(0.4, 1.0)
axes[0].set_xlabel("F1 (weighted)")
axes[0].set_title("German Sentiment (1,490 test)")
for i, v in enumerate(df_german["F1 (weighted)"]):
    axes[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=10)

# Yelp
colors_yelp = ["#10B981" if "BERT" in m else "#3B82F6" for m in df_yelp["Model"]]
axes[1].barh(df_yelp["Model"], df_yelp["F1 (weighted)"], color=colors_yelp)
axes[1].set_xlim(0.5, 0.85)
axes[1].set_xlabel("F1 (weighted)")
axes[1].set_title("Yelp Reviews (5,000 test)")
for i, v in enumerate(df_yelp["F1 (weighted)"]):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.savefig("../results/plots/f1_comparison_both_datasets.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Confusion Matrices — Top Models

In [ ]:
def plot_confusion_matrices(raw_data, title_prefix, models_to_plot=None):
    """Plot confusion matrices for selected models."""
    if models_to_plot is None:
        models_to_plot = list(raw_data.keys())[:4]
    
    n = len(models_to_plot)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    
    for ax, model_name in zip(axes, models_to_plot):
        data = raw_data.get(model_name)
        if not data:
            continue
        cm = data.get("test", {}).get("confusion_matrix") or data.get("confusion_matrix")
        if not cm:
            continue
        
        cm_array = np.array(cm)
        cm_norm = cm_array.astype(float) / cm_array.sum(axis=1, keepdims=True)
        
        sns.heatmap(
            cm_norm, annot=True, fmt=".1%", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=ax, vmin=0, vmax=1, cbar=False,
        )
        ax.set_title(model_name, fontsize=11)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    
    plt.suptitle(f"{title_prefix} — Normalized Confusion Matrices", fontsize=14, y=1.02)
    plt.tight_layout()
    return fig

# German: BERT vs SVM vs GPT-4o-mini
fig = plot_confusion_matrices(
    raw_german, "German Sentiment",
    ["BERT (gbert-base)", "SVM (TF-IDF)", "GPT-4o-mini (zero-shot)"]
)
fig.savefig("../results/plots/confusion_matrices_german.png", dpi=150, bbox_inches="tight")
plt.show()

# Yelp: BERT vs LogReg
fig = plot_confusion_matrices(
    raw_yelp, "Yelp Reviews",
    ["BERT (bert-base-uncased)", "LogReg (TF-IDF)", "Naive Bayes (TF-IDF)"]
)
fig.savefig("../results/plots/confusion_matrices_yelp.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-Class F1 Comparison

In [ ]:
def extract_per_class_f1(raw_data):
    """Extract per-class F1 scores for all models."""
    rows = []
    for model_name, data in raw_data.items():
        report = data.get("test_classification_report") or data.get("classification_report", {})
        if not report:
            continue
        for label in LABEL_NAMES:
            if label in report:
                rows.append({
                    "Model": model_name,
                    "Class": label,
                    "F1": report[label].get("f1-score", 0),
                })
    return pd.DataFrame(rows)

# German per-class
pc_german = extract_per_class_f1(raw_german)
if not pc_german.empty:
    fig = px.bar(
        pc_german, x="Model", y="F1", color="Class",
        barmode="group",
        color_discrete_map={"negative": "#EF4444", "neutral": "#F59E0B", "positive": "#10B981"},
        title="German Sentiment — Per-Class F1",
    )
    fig.update_layout(xaxis_tickangle=-45, height=450)
    fig.show()

# Yelp per-class
pc_yelp = extract_per_class_f1(raw_yelp)
if not pc_yelp.empty:
    fig = px.bar(
        pc_yelp, x="Model", y="F1", color="Class",
        barmode="group",
        color_discrete_map={"negative": "#EF4444", "neutral": "#F59E0B", "positive": "#10B981"},
        title="Yelp Reviews — Per-Class F1",
    )
    fig.update_layout(xaxis_tickangle=-45, height=450)
    fig.show()

## 5. Accuracy vs. Latency Trade-off

In [ ]:
# Combine both datasets for scatter plot
df_german_copy = df_german.copy()
df_german_copy["Dataset"] = "German"
df_yelp_copy = df_yelp.copy()
df_yelp_copy["Dataset"] = "Yelp"
df_combined = pd.concat([df_german_copy, df_yelp_copy], ignore_index=True)

fig = px.scatter(
    df_combined,
    x="Latency (ms)", y="F1 (weighted)",
    color="Dataset", symbol="Dataset",
    size="Accuracy",
    hover_name="Model",
    log_x=True,
    title="F1 vs. Latency (log scale) — All Models",
    labels={"Latency (ms)": "Latency (ms/sample, log scale)"},
)
fig.update_layout(height=500, width=900)
fig.show()

## 6. Learning Curve

In [ ]:
lc_path = METRICS_DIR / "learning_curve_svm_german.json"
if lc_path.exists():
    with open(lc_path) as f:
        lc_data = json.load(f)
    lc_df = pd.DataFrame(lc_data)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(lc_df["train_size"], lc_df["f1_weighted"], "o-", color="#3B82F6", linewidth=2, markersize=8)
    ax.fill_between(lc_df["train_size"], lc_df["f1_weighted"] - 0.01, lc_df["f1_weighted"] + 0.01, alpha=0.15, color="#3B82F6")
    ax.set_xlabel("Training Samples")
    ax.set_ylabel("F1 (weighted)")
    ax.set_title("SVM Learning Curve — German Sentiment")
    ax.set_ylim(0.75, 0.90)
    ax.axhline(y=lc_df["f1_weighted"].max(), color="gray", linestyle="--", alpha=0.5)
    ax.annotate(f"Peak: {lc_df['f1_weighted'].max():.3f}", 
                xy=(lc_df["train_size"].iloc[-1], lc_df["f1_weighted"].max()),
                fontsize=11, ha="right")
    plt.tight_layout()
    plt.savefig("../results/plots/learning_curve_svm.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No learning curve data found. Run: python run_training.py --model classical --learning-curve")

## 7. Cross-Dataset BERT Gap Analysis

In [ ]:
cross_data = pd.DataFrame([
    {"Dataset": "German", "Approach": "Fine-tuned BERT", "F1": 0.9119},
    {"Dataset": "German", "Approach": "Best Classical ML", "F1": 0.8562},
    {"Dataset": "German", "Approach": "Best LLM", "F1": 0.7517},
    {"Dataset": "Yelp", "Approach": "Fine-tuned BERT", "F1": 0.7607},
    {"Dataset": "Yelp", "Approach": "Best Classical ML", "F1": 0.7431},
])

fig = px.bar(
    cross_data, x="Dataset", y="F1", color="Approach",
    barmode="group",
    color_discrete_map={
        "Fine-tuned BERT": "#10B981",
        "Best Classical ML": "#3B82F6",
        "Best LLM": "#8B5CF6",
    },
    title="Cross-Dataset: BERT vs Classical ML vs LLM",
    text_auto=".3f",
)
fig.update_layout(height=450, yaxis_range=[0.4, 1.0])
fig.show()

print("\nBERT advantage over best classical:")
print(f"  German: +{(0.9119 - 0.8562)*100:.1f} percentage points")
print(f"  Yelp:   +{(0.7607 - 0.7431)*100:.1f} percentage points")
print(f"\n=> BERT's advantage is {(0.9119 - 0.8562) / (0.7607 - 0.7431):.1f}x larger on German than Yelp")

## 8. Summary Table

In [ ]:
print("=" * 70)
print("GERMAN SENTIMENT — Final Results (1,490 test samples)")
print("=" * 70)
display(df_german[["Model", "F1 (weighted)", "Accuracy", "Latency (ms)"]].reset_index(drop=True))

print()
print("=" * 70)
print("YELP REVIEWS — Final Results (5,000 test samples)")
print("=" * 70)
display(df_yelp[["Model", "F1 (weighted)", "Accuracy", "ROC-AUC", "Latency (ms)"]].reset_index(drop=True))